EVAL - Aprendizaje Profundo con YOLO
==========================================================

En este cuaderno realizaremos la evaluación sistemática de las cinco variantes de YOLOv8 (n, s, m, l, x) sobre nuestro subconjunto de validación (100 imágenes de COCO 2017).

El objetivo es extraer métricas cuantitativas para comparar el compromiso (trade-off) entre:
1.  **Precisión:** Medida a través del **F1-Score** (media armónica de precisión y exhaustividad).
2.  **Velocidad:** Medida a través del tiempo promedio de inferencia por imagen.

Los resultados se guardarán en un archivo JSON para su posterior análisis gráfico en `results.ipynb`.

In [ ]:
import os
import pandas as pd
import numpy as np
from ultralytics import YOLO
from pycocotools.coco import COCO
from tqdm import tqdm

# Rutas
IMG_DIR = 'datasets/coco/val2017'
ANN_FILE = 'datasets/coco/annotations/instances_val2017.json'
RESULTS_FILE = 'evaluation_results.json'

# Cargar Ground Truth (GT)
coco = COCO(ANN_FILE)
existing_imgs = set(os.listdir(IMG_DIR))
img_ids = [img_id for img_id in coco.getImgIds() 
           if coco.loadImgs(img_id)[0]['file_name'] in existing_imgs]

print(f"Imágenes listas para evaluar: {len(img_ids)}")

### Funciones de Métrica (IoU y F1)

In [ ]:
def calcular_iou(box1, box2):
    """Calcula Intersection over Union (IoU)"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - inter_area
    
    return inter_area / union_area if union_area > 0 else 0


def calcular_f1_batch(gt_boxes, gt_classes, pred_boxes, pred_classes, iou_thresh=0.5):
    """Calcula F1, Precision y Recall para una imagen."""
    if len(pred_boxes) == 0:
        return 0.0, 0.0, 0.0
    if len(gt_boxes) == 0:
        return 0.0, 0.0, 0.0

    tp = 0
    fp = 0
    matched_gt = set()

    for i, p_box in enumerate(pred_boxes):
        best_iou = 0
        best_gt_idx = -1
        
        for j, g_box in enumerate(gt_boxes):
            if j in matched_gt: continue
            if pred_classes[i] != gt_classes[j]: continue
            
            iou = calcular_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = j
        
        if best_iou >= iou_thresh:
            tp += 1
            matched_gt.add(best_gt_idx)
        else:
            fp += 1

    fn = len(gt_boxes) - len(matched_gt)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return f1, precision, recall

### Bucle de Evaluación

In [ ]:
models_list = ['n', 's', 'm', 'l', 'x']
final_results = []

print("Iniciando evaluación masiva...")

for variant in models_list:
    model_name = f'yolov8{variant}.pt'
    print(f"\n--- Evaluando Modelo: {model_name} ---")
    
    try:
        model = YOLO(model_name)
    except Exception as e:
        print(f"Error cargando {model_name}: {e}")
        continue
        
    f1_scores, precisions, recalls, inference_times = [], [], [], []
    
    # Iterar sobre las imágenes
    for img_id in tqdm(img_ids, desc=f"Procesando {variant}"):
        img_info = coco.loadImgs(img_id)[0]
        img_path = os.path.join(IMG_DIR, img_info['file_name'])
        
        # Obtener GT
        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)
        gt_boxes = [[ann['bbox'][0], ann['bbox'][1], ann['bbox'][0]+ann['bbox'][2], ann['bbox'][1]+ann['bbox'][3]] for ann in anns]
        gt_classes = [coco.loadCats(ann['category_id'])[0]['name'] for ann in anns]
            
        # Inferencia
        results = model.predict(img_path, conf=0.25, verbose=False)
        result = results[0]
        
        pred_boxes = result.boxes.xyxy.cpu().numpy()
        pred_classes = [result.names[int(c)] for c in result.boxes.cls.cpu().numpy()]
        
        f1, prec, rec = calcular_f1_batch(gt_boxes, gt_classes, pred_boxes, pred_classes)
        
        f1_scores.append(f1)
        precisions.append(prec)
        recalls.append(rec)
        inference_times.append(result.speed['inference'])
        
    avg_f1 = np.mean(f1_scores)
    avg_time = np.mean(inference_times)
    
    n_params = sum(p.numel() for p in model.model.parameters())

    print(f"Resultados {variant.upper()}: F1={avg_f1:.3f} | Tiempo={avg_time:.1f}ms")
    
    final_results.append({
        "model": f"YOLOv8{variant}",
        "variant": variant,
        "f1": round(avg_f1, 4),
        "precision": round(np.mean(precisions), 4),
        "recall": round(np.mean(recalls), 4),
        "time_ms": round(avg_time, 2),
        "params": n_params
    })

print("\nEvaluación completada.")

### Guardar Resultados

In [ ]:
# Guardar resultados en CSV dentro de la carpeta 'resultados'
output_dir = 'resultados'
os.makedirs(output_dir, exist_ok=True)
df_results = pd.DataFrame(final_results)

# Reordenar columnas
column_order = ['model', 'variant', 'f1', 'precision', 'recall', 'time_ms', 'params']
df_results = df_results[column_order]

# Definir ruta completa
csv_filename = os.path.join(output_dir, 'evaluation_results.csv')
df_results.to_csv(csv_filename, index=False)

print(f"Resultados guardados exitosamente en: {csv_filename}")

print("\n--- Tabla Final de Resultados ---")
print(df_results)